# AI Math Tutor: QLoRA Fine-Tuning on Free GPU (Colab / Kaggle)

This notebook fine-tunes **Qwen2.5-Math-1.5B-Instruct** (or `Qwen2.5-1.5B-Instruct`) using **QLoRA (4-bit)** to act as an expert step-by-step math tutor.
Optimized to run within the free 15–16 GB GPU limit (Tesla T4 on Colab or P100/T4x2 on Kaggle).

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Install required dependencies
!pip install -q --upgrade pip
!pip install -q torch transformers peft trl datasets bitsandbytes accelerate kagglehub sympy pyyaml

In [ ]:
# 3. Prepare dataset (Download from Kaggle or load existing)
import os
import kagglehub
from pathlib import Path

# Download the 520K dataset via kagglehub
print("Downloading dataset...")
path = kagglehub.dataset_download("dineshkumar0705/large-scale-math-reasoning-dataset-520k")
print("Dataset downloaded to:", path)

# Run data preparation script
!python -m tutor.data.prepare_dataset --input-path "{path}" --max-samples 50000 --output-dir "tutor/data/processed"

In [ ]:
# 4. Run QLoRA Fine-Tuning
from tutor.training.finetune_lora import train_lora

adapter_path = train_lora(
    config_path="tutor/training/config.yaml",
    override_output_dir="models/qwen2.5-math-tutor-lora",
    override_epochs=3
)
print("Training complete! Adapter weights saved to:", adapter_path)

In [ ]:
# 5. Test Inference with Fine-Tuned Model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"
adapter_dir = "models/qwen2.5-math-tutor-lora"

print("Loading base model & adapter...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)
model = PeftModel.from_pretrained(base_model, adapter_dir)

sample_problem = "Solve for x: 3*x + 7 = 22"
messages = [
    {"role": "system", "content": "You are an expert, patient AI Math Tutor. Explain step by step, never skip reasoning."},
    {"role": "user", "content": sample_problem}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.2)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== Tutor Response ===")
print(response)